In [1]:
import json
import os
import pickle
import time
import zipfile
import numpy as np
import pandas as pd
from datetime import datetime
import random
import torch
import tempfile
import shutil
from itertools import product


import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, hinge_loss,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
# To add LogisticRegression: from sklearn.linear_model import LogisticRegression
# To add NB-SVM: from scipy.sparse import csr_matrix


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

SEED=42
 
set_seed(SEED)

sns.set_style("whitegrid")
print("Libraries imported successfully.")

DATASET = "IMDB"
MODEL = "SVM"
VECTORIZER = "tfidf"  # "tfidf" or "bow"
N_ITERATIONS = 200
TEST_LABELS = True if DATASET == "IMDB" else False
SEED = 42

STRIP_ACCENTS = "unicode"

# Grid search parameter lists
C_values = [0.1, 0.5, 1]
ngram_values = [(1, 2), (1, 3)]
max_features_values = [200000, 300000]
sublinear_values = [True, False] if VECTORIZER == "tfidf" else [None]

TEXT_COLUMN = "sentence_preprocessed" if DATASET == "SST-2" else "text_preprocessed"
LABEL_COLUMN = "label"
ROOT_DIR = os.getcwd()

print(f"Dataset:       {DATASET}")
print(f"Vectorizer:    {VECTORIZER}")
print(f"Max iterations:{N_ITERATIONS}")
print(f"Strip accents: {STRIP_ACCENTS}")
print(f"Test labels:   {TEST_LABELS}")
print(f"Seed:          {SEED}")
print(f"Text column:   {TEXT_COLUMN}")
print(f"Label column:  {LABEL_COLUMN}")
print(f"Root dir:      {ROOT_DIR}")
print(f"\nGrid search space:")
print(f"  C:            {C_values}")
print(f"  Ngram:        {ngram_values}")
print(f"  Max features: {max_features_values}")
if VECTORIZER == "tfidf":
    print(f"  Sublinear TF: {sublinear_values}")
print(f"  Total combos: {len(C_values) * len(ngram_values) * len(max_features_values) * len(sublinear_values)}")


Libraries imported successfully.
Dataset:       IMDB
Vectorizer:    tfidf
Max iterations:200
Strip accents: unicode
Test labels:   True
Seed:          42
Text column:   text_preprocessed
Label column:  label
Root dir:      c:\Users\cemil\OneDrive\Desktop\NLP

Grid search space:
  C:            [0.1, 0.5, 1]
  Ngram:        [(1, 2), (1, 3)]
  Max features: [200000, 300000]
  Sublinear TF: [True, False]
  Total combos: 24


In [2]:
dataset_path = os.path.join(ROOT_DIR, DATASET)

def load_split(split_name):
    file_path = os.path.join(dataset_path, f"{split_name}.json")
    if not os.path.exists(file_path):
        print(f"  {split_name}.json not found in {dataset_path}")
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"  {split_name}.json: {len(df)} samples")
    return df

print(f"Loading data from: {dataset_path}\n")
df_train = load_split("train")
df_validation = load_split("validation")
df_test = load_split("test")

if df_train is not None:
    print(f"\nFirst 3 train examples:")
    display(df_train.head(3))

Loading data from: c:\Users\cemil\OneDrive\Desktop\NLP\IMDB

  train.json: 20000 samples
  validation.json: 5000 samples
  test.json: 25000 samples

First 3 train examples:


,text_original,text_preprocessed,label
0,I have always been a huge James Bond fanatic! ...,i have always been a huge james bond fanatic! ...,1
1,I am a Christian and I say this movie had terr...,i am a christian and i say this movie had terr...,0
2,"Neatly sandwiched between THE STRANGER, a smal...","neatly sandwiched between the stranger, a smal...",1


In [3]:
X_train_text = df_train[TEXT_COLUMN].fillna("").values
y_train = df_train[LABEL_COLUMN].values

X_val_text = df_validation[TEXT_COLUMN].fillna("").values
y_val = df_validation[LABEL_COLUMN].values

X_test_text = df_test[TEXT_COLUMN].fillna("").values
y_test = None
if TEST_LABELS and LABEL_COLUMN in df_test.columns:
    y_test = df_test[LABEL_COLUMN].values

print(f"Train samples: {len(X_train_text)}")
print(f"Val samples:   {len(X_val_text)}")
print(f"Test samples:  {len(X_test_text)}")
print(f"Unique classes: {np.unique(y_train)}")

Train samples: 20000
Val samples:   5000
Test samples:  25000
Unique classes: [0 1]


In [4]:
# To add LogisticRegression: add an elif branch returning LogisticRegression(C=C, ...) with loss_type="log_loss"
# To add NB-SVM: add get_nbsvm_features() for log-count ratios + an elif branch using LinearSVC on NB-weighted features

def build_model(model_name, C, max_iter):
    if model_name == "SVM":
        model = LinearSVC(C=C, max_iter=max_iter, loss="squared_hinge", random_state=SEED)
        params = {"type": "LinearSVC", "C": C, "max_iter": max_iter, "loss": "squared_hinge"}
        loss_type = "squared_hinge"
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model, params, loss_type

def compute_loss(model, X, y, loss_type):
    decision = model.decision_function(X)

    if loss_type == "hinge":
        return hinge_loss(y, decision)

    elif loss_type == "squared_hinge":
        y_signed = np.where(y == 1, 1, -1)
        margins = 1 - y_signed * decision
        return np.mean(np.maximum(0, margins) ** 2)

    else:
        raise ValueError(f"Unsupported loss type: {loss_type}")
    # To add log_loss: elif loss_type == "log_loss": return sk_log_loss(y, model.predict_proba(X))

print("build_model() and compute_loss() ready.")

build_model() and compute_loss() ready.


In [5]:
# ============================================================
# Grid Search over C, ngram, max_features, sublinear_tf
# Compute ALL validation metrics per combo, keep the best model
# ============================================================

combos = list(product(C_values, ngram_values, max_features_values, sublinear_values))
print(f"Grid search: {len(combos)} combinations, max_iter={N_ITERATIONS}\n")

best_gs_acc = -1
best_gs_loss = float('inf')
best_gs_params = None
best_gs_metrics = None
best_gs_model = None
best_gs_tfidf = None
best_gs_train_time = 0
gs_results = []

start_gs = time.time()

for idx, (c, ngram, max_feat, sublinear) in enumerate(combos, 1):
    if VECTORIZER == "tfidf":
        tfidf_gs = TfidfVectorizer(
            max_features=max_feat,
            ngram_range=ngram,
            sublinear_tf=sublinear,
            strip_accents=STRIP_ACCENTS
        )
    else:
        tfidf_gs = CountVectorizer(
            max_features=max_feat,
            ngram_range=ngram,
            strip_accents=STRIP_ACCENTS
        )
    X_tr = tfidf_gs.fit_transform(X_train_text)
    X_vl = tfidf_gs.transform(X_val_text)

    mdl, mdl_params, lt = build_model(MODEL, c, N_ITERATIONS)
    t0 = time.time()
    mdl.fit(X_tr, y_train)
    t_train = time.time() - t0

    y_tr_pred = mdl.predict(X_tr)
    y_vl_pred = mdl.predict(X_vl)

    # Train metrics
    tr_acc = accuracy_score(y_train, y_tr_pred)
    tr_loss = compute_loss(mdl, X_tr, y_train, lt)

    # Validation metrics
    vl_acc = accuracy_score(y_val, y_vl_pred)
    vl_loss = compute_loss(mdl, X_vl, y_val, lt)
    vl_f1 = f1_score(y_val, y_vl_pred, average="macro", zero_division=0)
    vl_prec = precision_score(y_val, y_vl_pred, average="macro", zero_division=0)
    vl_rec = recall_score(y_val, y_vl_pred, average="macro", zero_division=0)
    vl_prec_pc = precision_score(y_val, y_vl_pred, average=None, zero_division=0).tolist()
    vl_rec_pc = recall_score(y_val, y_vl_pred, average=None, zero_division=0).tolist()
    vl_f1_pc = f1_score(y_val, y_vl_pred, average=None, zero_division=0).tolist()
    vl_cm = confusion_matrix(y_val, y_vl_pred).tolist()
    vl_dec = mdl.decision_function(X_vl)
    vl_fpr, vl_tpr, _ = roc_curve(y_val, vl_dec)
    vl_auc = auc(vl_fpr, vl_tpr)
    vl_pr_prec, vl_pr_rec, _ = precision_recall_curve(y_val, vl_dec)
    vl_avg_prec = average_precision_score(y_val, vl_dec)

    combo_metrics = {
        "C": c, "ngram": ngram, "max_features": max_feat, "sublinear_tf": sublinear,
        "model_params": mdl_params, "loss_type": lt, "train_time": t_train,
        "train_accuracy": tr_acc, "train_loss": tr_loss,
        "val_acc": vl_acc, "val_loss": vl_loss,
        "val_f1_macro": vl_f1, "val_precision_macro": vl_prec, "val_recall_macro": vl_rec,
        "val_precision_per_class": vl_prec_pc, "val_recall_per_class": vl_rec_pc,
        "val_f1_per_class": vl_f1_pc, "val_conf_matrix": vl_cm,
        "val_fpr": vl_fpr, "val_tpr": vl_tpr, "val_auc": vl_auc,
        "val_pr_precision": vl_pr_prec, "val_pr_recall": vl_pr_rec, "val_avg_precision": vl_avg_prec,
    }
    gs_results.append(combo_metrics)

    if vl_acc > best_gs_acc or (vl_acc == best_gs_acc and vl_loss < best_gs_loss):
        best_gs_acc = vl_acc
        best_gs_loss = vl_loss
        best_gs_params = {"C": c, "ngram": ngram, "max_features": max_feat, "sublinear_tf": sublinear}
        best_gs_metrics = combo_metrics
        best_gs_model = pickle.loads(pickle.dumps(mdl))
        best_gs_tfidf = pickle.loads(pickle.dumps(tfidf_gs))
        best_gs_train_time = t_train

    print(f"  [{idx:3d}/{len(combos)}] C={c}, ngram={ngram}, max_feat={max_feat}, "
          f"sub_tf={str(sublinear):5s} => Val Acc: {vl_acc:.4f} | Val Loss: {vl_loss:.4f} | Val F1: {vl_f1:.4f}")

gs_time = time.time() - start_gs

# Sort results by val_acc desc, then val_loss asc
gs_results_sorted = sorted(gs_results, key=lambda x: (-x["val_acc"], x["val_loss"]))

print(f"\nGrid search complete in {gs_time:.1f}s")
print(f"\n{'='*90}")
print(f"  ALL COMBINATIONS (sorted by Val Acc desc, Val Loss asc):")
print(f"{'='*90}")
print(f"  {'#':>3}  {'C':>5}  {'Ngram':>8}  {'MaxFeat':>8}  {'SubTF':>5}  {'Val Acc':>8}  {'Val Loss':>9}  {'Val F1':>7}")
print(f"  {'-'*84}")
for i, r in enumerate(gs_results_sorted, 1):
    marker = " <-- BEST" if (r["C"] == best_gs_params["C"] and r["ngram"] == best_gs_params["ngram"]
                             and r["max_features"] == best_gs_params["max_features"]
                             and r["sublinear_tf"] == best_gs_params["sublinear_tf"]) else ""
    print(f"  {i:3d}  {r['C']:>5}  {str(r['ngram']):>8}  {r['max_features']:>8}  {str(r['sublinear_tf']):>5}  "
          f"{r['val_acc']:.4f}    {r['val_loss']:.4f}   {r['val_f1_macro']:.4f}{marker}")

# Set best params + model + tfidf
C_VALUE = best_gs_params["C"]
NGRAM_RANGE = best_gs_params["ngram"]
MAX_FEATURES = best_gs_params["max_features"]
SUBLINEAR_TF = best_gs_params["sublinear_tf"]

model = best_gs_model
tfidf = best_gs_tfidf
model_params = best_gs_metrics["model_params"]
loss_type = best_gs_metrics["loss_type"]
train_time = best_gs_train_time
entries_per_second = len(y_train) / train_time
best_iter = N_ITERATIONS

print(f"\n{'='*90}")
print(f"  BEST PARAMS SELECTED:")
print(f"  C={C_VALUE}, ngram={NGRAM_RANGE}, max_features={MAX_FEATURES}, sublinear_tf={SUBLINEAR_TF}")
print(f"  Val Acc: {best_gs_acc:.4f} | Val Loss: {best_gs_loss:.4f} | Val F1: {best_gs_metrics['val_f1_macro']:.4f}")
print(f"  Val AUC: {best_gs_metrics['val_auc']:.4f} | Train Acc: {best_gs_metrics['train_accuracy']:.4f}")
print(f"{'='*90}")


Grid search: 24 combinations, max_iter=200

  [  1/24] C=0.1, ngram=(1, 2), max_feat=200000, sub_tf=True  => Val Acc: 0.8994 | Val Loss: 0.4084 | Val F1: 0.8994
  [  2/24] C=0.1, ngram=(1, 2), max_feat=200000, sub_tf=False => Val Acc: 0.8964 | Val Loss: 0.4129 | Val F1: 0.8964
  [  3/24] C=0.1, ngram=(1, 2), max_feat=300000, sub_tf=True  => Val Acc: 0.8988 | Val Loss: 0.4144 | Val F1: 0.8988
  [  4/24] C=0.1, ngram=(1, 2), max_feat=300000, sub_tf=False => Val Acc: 0.8966 | Val Loss: 0.4182 | Val F1: 0.8966
  [  5/24] C=0.1, ngram=(1, 3), max_feat=200000, sub_tf=True  => Val Acc: 0.8988 | Val Loss: 0.4165 | Val F1: 0.8988
  [  6/24] C=0.1, ngram=(1, 3), max_feat=200000, sub_tf=False => Val Acc: 0.8972 | Val Loss: 0.4193 | Val F1: 0.8972
  [  7/24] C=0.1, ngram=(1, 3), max_feat=300000, sub_tf=True  => Val Acc: 0.8986 | Val Loss: 0.4256 | Val F1: 0.8986
  [  8/24] C=0.1, ngram=(1, 3), max_feat=300000, sub_tf=False => Val Acc: 0.8944 | Val Loss: 0.4273 | Val F1: 0.8944
  [  9/24] C=0.5, ng

In [6]:
# ============================================================
# Set up final matrices + test predictions using best model from grid search
# ============================================================

X_train_final = tfidf.transform(X_train_text)
X_val_final = tfidf.transform(X_val_text)
X_test_final = tfidf.transform(X_test_text)

print(f"{VECTORIZER.upper()} vocabulary: {len(tfidf.vocabulary_)} features")
print(f"X_train shape: {X_train_final.shape}")
print(f"X_val shape:   {X_val_final.shape}")
print(f"X_test shape:  {X_test_final.shape}")

# Reuse pre-computed validation metrics from grid search
m = best_gs_metrics
train_accuracy = m["train_accuracy"]
train_loss_final = m["train_loss"]
val_accuracy = m["val_acc"]
val_loss_final = m["val_loss"]
val_f1_macro = m["val_f1_macro"]
val_precision_macro = m["val_precision_macro"]
val_recall_macro = m["val_recall_macro"]
val_precision_per_class = m["val_precision_per_class"]
val_recall_per_class = m["val_recall_per_class"]
val_f1_per_class = m["val_f1_per_class"]
val_conf_matrix = m["val_conf_matrix"]
val_fpr = m["val_fpr"]
val_tpr = m["val_tpr"]
val_auc = m["val_auc"]
val_pr_precision = m["val_pr_precision"]
val_pr_recall = m["val_pr_recall"]
val_avg_precision = m["val_avg_precision"]
class_labels = sorted(np.unique(y_val).tolist())

# Predictions
y_train_pred_final = model.predict(X_train_final)
y_val_pred = model.predict(X_val_final)
y_test_pred = model.predict(X_test_final)

# Only compute test metrics if labels available (not already computed in grid search)
if TEST_LABELS:
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision_macro = precision_score(y_test, y_test_pred, average="macro", zero_division=0)
    test_recall_macro = recall_score(y_test, y_test_pred, average="macro", zero_division=0)
    test_f1_macro = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    test_precision_per_class = precision_score(y_test, y_test_pred, average=None, zero_division=0).tolist()
    test_recall_per_class = recall_score(y_test, y_test_pred, average=None, zero_division=0).tolist()
    test_f1_per_class = f1_score(y_test, y_test_pred, average=None, zero_division=0).tolist()
    test_conf_matrix = confusion_matrix(y_test, y_test_pred).tolist()
    test_loss_final = compute_loss(model, X_test_final, y_test, loss_type)
    test_decision_scores = model.decision_function(X_test_final)
    test_fpr, test_tpr, _ = roc_curve(y_test, test_decision_scores)
    test_auc = auc(test_fpr, test_tpr)
    test_pr_precision, test_pr_recall, _ = precision_recall_curve(y_test, test_decision_scores)
    test_avg_precision = average_precision_score(y_test, test_decision_scores)

val_decision_scores = model.decision_function(X_val_final)

print(f"\n{'='*60}")
print(f"  RESULTS {MODEL} on {DATASET}")
print(f"{'='*60}")
print(f"  Train Accuracy:      {train_accuracy:.4f}   | Loss: {train_loss_final:.4f}")
print(f"  Val Accuracy:        {val_accuracy:.4f}   | Loss: {val_loss_final:.4f}")
print(f"  Val F1 (macro):      {val_f1_macro:.4f}")
print(f"  Val AUC:             {val_auc:.4f}")

if not TEST_LABELS:
    unique, counts = np.unique(y_test_pred, return_counts=True)
    print(f"  Test prediction distribution: {dict(zip(unique.tolist(), counts.tolist()))}")

print(f"\n  Classification Report (VALIDATION):")
print(classification_report(y_val, y_val_pred, zero_division=0))


TFIDF vocabulary: 200000 features
X_train shape: (20000, 200000)
X_val shape:   (5000, 200000)
X_test shape:  (25000, 200000)

  RESULTS SVM on IMDB
  Train Accuracy:      0.9999   | Loss: 0.0506
  Val Accuracy:        0.9128   | Loss: 0.3056
  Val F1 (macro):      0.9128
  Val AUC:             0.9703

  Classification Report (VALIDATION):
              precision    recall  f1-score   support

           0       0.92      0.91      0.91      2500
           1       0.91      0.92      0.91      2500

    accuracy                           0.91      5000
   macro avg       0.91      0.91      0.91      5000
weighted avg       0.91      0.91      0.91      5000



In [7]:
if TEST_LABELS:
    cm_array = np.array(test_conf_matrix)
    cm_labels = sorted(np.unique(y_test).tolist())
else:
    cm_array = np.array(val_conf_matrix)
    cm_labels = sorted(np.unique(y_val).tolist())

fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_array, annot=True, fmt="d", cmap="Blues",
            xticklabels=cm_labels, yticklabels=cm_labels, ax=ax_cm)
ax_cm.set_xlabel("Predicted")
ax_cm.set_ylabel("Actual")
ax_cm.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
fig_cm.tight_layout()
plt.show()

print(f"Classes: {cm_labels}")
print(f"Matrix:\n{cm_array}")

Classes: [0, 1]
Matrix:
[[11257  1243]
 [ 1230 11270]]


C:\Users\cemil\AppData\Local\Temp\ipykernel_26348\3610909509.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

if TEST_LABELS:
    ax_roc.plot(test_fpr, test_tpr, linewidth=2, color="#1f77b4",
                label=f"Test ROC (AUC = {test_auc:.4f})")
else:
    ax_roc.plot(val_fpr, val_tpr, linewidth=2, color="#1f77b4",
                label=f"Validation ROC (AUC = {val_auc:.4f})")

ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Random")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title(f"{MODEL} - {DATASET} | ROC Curve", fontweight="bold", color="black")
ax_roc.legend(loc="lower right", fontsize=12)
ax_roc.grid(True, alpha=0.3)
fig_roc.tight_layout()
plt.show()

C:\Users\cemil\AppData\Local\Temp\ipykernel_26348\3758031253.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
fig_pr, ax_pr = plt.subplots(figsize=(8, 6))

if TEST_LABELS:
    ax_pr.plot(test_pr_recall, test_pr_precision, linewidth=2, color="#1f77b4",
               label=f"Test PR (AP = {test_avg_precision:.4f})")
else:
    ax_pr.plot(val_pr_recall, val_pr_precision, linewidth=2, color="#1f77b4",
               label=f"Validation PR (AP = {val_avg_precision:.4f})")

ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_pr.legend(loc="lower left", fontsize=12)
ax_pr.grid(True, alpha=0.3)
ax_pr.set_xlim([0, 1])
ax_pr.set_ylim([0, 1.05])
fig_pr.tight_layout()
plt.show()

C:\Users\cemil\AppData\Local\Temp\ipykernel_26348\1345400219.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
if TEST_LABELS:
    pred_source = y_test_pred
    source_label = "Test"
else:
    pred_source = y_val_pred
    source_label = "Validation"

counts = [np.sum(pred_source == 0), np.sum(pred_source == 1)]
labels_bar = ["Negative (0)", "Positive (1)"]
colors = ["#e74c3c", "#2ecc71"]

fig_dist, ax_dist = plt.subplots(figsize=(6, 5))
bars = ax_dist.bar(labels_bar, counts, color=colors, edgecolor="black", width=0.5)
for bar, count in zip(bars, counts):
    ax_dist.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                 str(count), ha="center", va="bottom", fontweight="bold", fontsize=12)
ax_dist.set_xlabel("Predicted Sentiment")
ax_dist.set_ylabel("Number of Samples")
ax_dist.set_title(f"{MODEL} - {DATASET}", fontweight="bold")
fig_dist.tight_layout()
plt.show()

print(f"Negative: {counts[0]}, Positive: {counts[1]}")


Negative: 12487, Positive: 12513


C:\Users\cemil\AppData\Local\Temp\ipykernel_26348\185131405.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
TOP_N = 40

feature_names = tfidf.get_feature_names_out()
coefs = model.coef_[0]

top_pos_idx = np.argsort(coefs)[-TOP_N:]
top_neg_idx = np.argsort(coefs)[:TOP_N]

fig_words, (ax_neg, ax_pos) = plt.subplots(1, 2, figsize=(14, 6))

ax_neg.barh(feature_names[top_neg_idx], coefs[top_neg_idx], color="#e74c3c")
ax_neg.set_xlabel("Coefficient Value")
ax_neg.set_title(f"Top {TOP_N} Negative Words", fontweight="bold")
ax_neg.invert_yaxis()

ax_pos.barh(feature_names[top_pos_idx], coefs[top_pos_idx], color="#2ecc71")
ax_pos.set_xlabel("Coefficient Value")
ax_pos.set_title(f"Top {TOP_N} Positive Words", fontweight="bold")

fig_words.suptitle(f"{MODEL} - {DATASET}", fontweight="bold", fontsize=14)
fig_words.tight_layout()
plt.show()

C:\Users\cemil\AppData\Local\Temp\ipykernel_26348\4242763087.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
def save_results(model, model_params, loss_type, tfidf, figures,
                 train_results, val_results, test_results,
                 train_time, entries_per_second,
                 dataset, model_name, c_value, n_iterations, best_iteration, test_labels,
                 text_column, label_column, class_labels,
                 df_train, df_validation, df_test, root_dir):

    model_bytes = pickle.dumps(model)
    model_size_bytes = len(model_bytes)
    model_size_mb = model_size_bytes / (1024 * 1024)

    timestamp_str = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
    run_name = f"C{c_value}_iters{n_iterations}_loss-{loss_type}_{timestamp_str}"

    output_dir = os.path.join(root_dir, model_name, dataset, run_name)
    os.makedirs(output_dir, exist_ok=True)

    for fig_name, fig_obj in figures.items():
        fig_obj.savefig(os.path.join(output_dir, fig_name), dpi=150, bbox_inches="tight")
    print(f"Charts saved: {list(figures.keys())}")

    tfidf_params = tfidf.get_params()
    metrics = {
        "model": model_name,
        "dataset": dataset,
        "run_name": run_name,
        "timestamp": timestamp_str,
        "model_params": model_params,
        "seed": SEED,
        "model_size": {
            "bytes": model_size_bytes,
            "MB": round(model_size_mb, 4)
        },
        "training": {
            "training_time_seconds": round(train_time, 4),
            "entries_per_second": round(entries_per_second, 2),
            "num_iterations": n_iterations,
            "best_iteration": best_iteration,
            "train_samples": len(df_train),
        },
        "vectorization": {
            "method": VECTORIZER.upper(),
            "max_features": tfidf_params.get("max_features"),
            "ngram_range": list(tfidf_params.get("ngram_range", ())),
            "sublinear_tf": tfidf_params.get("sublinear_tf"),
            "vocabulary_size": len(tfidf.vocabulary_)
        },
        "data": {
            "train_samples": len(df_train),
            "validation_samples": len(df_validation),
            "test_samples": len(df_test) if df_test is not None else 0,
            "text_column": text_column,
            "label_column": label_column,
            "classes": class_labels
        },
        "train_results": train_results,
        "validation_results": val_results,
    }

    if test_labels:
        metrics["test_results"] = test_results

    metrics_path = os.path.join(output_dir, "metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"metrics.json saved in: {output_dir}")
    return output_dir, metrics_path, run_name


figures = {
    "confusion_matrix.png": fig_cm,
    "roc_curve.png": fig_roc,
    "pr_curve.png": fig_pr,
    "prediction_distribution.png": fig_dist,
    "top_words.png": fig_words,
}

train_res = {
    "accuracy": round(train_accuracy, 4),
    "loss": round(train_loss_final, 4),
}

val_res = {
    "accuracy": round(val_accuracy, 4),
    "loss": round(val_loss_final, 4),
    "precision_macro": round(val_precision_macro, 4),
    "recall_macro": round(val_recall_macro, 4),
    "f1_macro": round(val_f1_macro, 4),
    "auc": round(val_auc, 4),
    "avg_precision": round(val_avg_precision, 4),
    "precision_per_class": [round(p, 4) for p in val_precision_per_class],
    "recall_per_class": [round(r, 4) for r in val_recall_per_class],
    "f1_per_class": [round(f, 4) for f in val_f1_per_class],
    "confusion_matrix": val_conf_matrix,
}

test_res = None
if TEST_LABELS:
    test_res = {
        "accuracy": round(test_accuracy, 4),
        "loss": round(test_loss_final, 4),
        "precision_macro": round(test_precision_macro, 4),
        "recall_macro": round(test_recall_macro, 4),
        "f1_macro": round(test_f1_macro, 4),
        "auc": round(test_auc, 4),
        "avg_precision": round(test_avg_precision, 4),
        "precision_per_class": [round(p, 4) for p in test_precision_per_class],
        "recall_per_class": [round(r, 4) for r in test_recall_per_class],
        "f1_per_class": [round(f, 4) for f in test_f1_per_class],
        "confusion_matrix": test_conf_matrix,
    }

output_dir, metrics_path, run_name = save_results(
    model=model, model_params=model_params, loss_type=loss_type, tfidf=tfidf,
    figures=figures,
    train_results=train_res, val_results=val_res, test_results=test_res,
    train_time=train_time, entries_per_second=entries_per_second,
    dataset=DATASET, model_name=MODEL, c_value=C_VALUE, n_iterations=N_ITERATIONS,
    best_iteration=best_iter,
    test_labels=TEST_LABELS, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN,
    class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test,
    root_dir=ROOT_DIR,

)

Charts saved: ['confusion_matrix.png', 'roc_curve.png', 'pr_curve.png', 'prediction_distribution.png', 'top_words.png']
metrics.json saved in: c:\Users\cemil\OneDrive\Desktop\NLP\SVM\IMDB\C1_iters200_loss-squared_hinge_14_04_2026_14_42_29


In [13]:
with open(metrics_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"{'='*60}")
print(f"  RUN SUMMARY: {run_name}")
print(f"{'='*60}")
print(f"  Model:              {saved['model']}")
print(f"  Dataset:            {saved['dataset']}")
print(f"  Training time:      {saved['training']['training_time_seconds']}s")
print(f"  Entries/sec:        {saved['training']['entries_per_second']}")
print(f"  Best iteration:     {saved['training']['best_iteration']}/{saved['training']['num_iterations']}")
print(f"  Model size:         {saved['model_size']['MB']} MB")
print(f"  Train Accuracy:     {saved['train_results']['accuracy']}")
print(f"  Train Loss:         {saved['train_results']['loss']}")
print(f"  Val Accuracy:       {saved['validation_results']['accuracy']}")
print(f"  Val Loss:           {saved['validation_results']['loss']}")
print(f"  Val F1 (macro):     {saved['validation_results']['f1_macro']}")
if "test_results" in saved:
    print(f"  Test:               saved in metrics.json")
else:
    print(f"  Test:               N/A (TEST_LABELS=False)")
print(f"{'='*60}")
print(f"\n  Files saved in: {output_dir}")
for f_name in os.listdir(output_dir):
    f_size = os.path.getsize(os.path.join(output_dir, f_name))
    print(f"    {f_name} ({f_size:,} bytes)")

  RUN SUMMARY: C1_iters200_loss-squared_hinge_14_04_2026_14_42_29
  Model:              SVM
  Dataset:            IMDB
  Training time:      0.4986s
  Entries/sec:        40110.89
  Best iteration:     200/200
  Model size:         1.5265 MB
  Train Accuracy:     0.9999
  Train Loss:         0.0506
  Val Accuracy:       0.9128
  Val Loss:           0.3056
  Val F1 (macro):     0.9128
  Test:               saved in metrics.json

  Files saved in: c:\Users\cemil\OneDrive\Desktop\NLP\SVM\IMDB\C1_iters200_loss-squared_hinge_14_04_2026_14_42_29
    confusion_matrix.png (24,577 bytes)
    metrics.json (2,080 bytes)
    prediction_distribution.png (30,241 bytes)
    pr_curve.png (35,874 bytes)
    roc_curve.png (60,058 bytes)
    top_words.png (138,475 bytes)


In [14]:
print(json.dumps(saved, indent=2, ensure_ascii=False))

{
  "model": "SVM",
  "dataset": "IMDB",
  "run_name": "C1_iters200_loss-squared_hinge_14_04_2026_14_42_29",
  "timestamp": "14_04_2026_14_42_29",
  "model_params": {
    "type": "LinearSVC",
    "C": 1,
    "max_iter": 200,
    "loss": "squared_hinge"
  },
  "seed": 42,
  "model_size": {
    "bytes": 1600611,
    "MB": 1.5265
  },
  "training": {
    "training_time_seconds": 0.4986,
    "entries_per_second": 40110.89,
    "num_iterations": 200,
    "best_iteration": 200,
    "train_samples": 20000
  },
  "vectorization": {
    "method": "TFIDF",
    "max_features": 200000,
    "ngram_range": [
      1,
      3
    ],
    "sublinear_tf": false,
    "vocabulary_size": 200000
  },
  "data": {
    "train_samples": 20000,
    "validation_samples": 5000,
    "test_samples": 25000,
    "text_column": "text_preprocessed",
    "label_column": "label",
    "classes": [
      0,
      1
    ]
  },
  "train_results": {
    "accuracy": 0.9999,
    "loss": 0.0506
  },
  "validation_results": {
    

In [15]:
import re as _re

if DATASET == "IMDB" and TEST_LABELS:
    label_map = {0: "Negative", 1: "Positive"}

    # Find all misclassified test examples
    misclassified_idx = np.where(y_test != y_test_pred)[0]
    print(f"Total misclassified: {len(misclassified_idx)} / {len(y_test)} "
          f"({100 * len(misclassified_idx) / len(y_test):.2f}%)\n")

    np.random.seed(SEED)
    selected_idx = np.random.choice(misclassified_idx, min(60, len(misclassified_idx)), replace=False)

    TEXT_ORIG_COL = "text_original" if "text_original" in df_test.columns else TEXT_COLUMN

    print(f"{'#':>3}  {'Predicted':>10}  {'Actual':>10}  Text (first ~35 words)")
    print(f"{'-' * 95}")
    for i, idx in enumerate(selected_idx):
        orig = str(df_test.iloc[idx][TEXT_ORIG_COL])
        orig_clean = _re.sub(r"<br\s*/?\s*>", " ", orig)
        short = " ".join(orig_clean.split()[:35]) + ("..." if len(orig_clean.split()) > 35 else "")
        print(f"{i+1:3d}  {label_map[y_test_pred[idx]]:>10}  {label_map[y_test[idx]]:>10}  {short}")
else:
    selected_idx = []
    print("Interpretability analysis: skipped (requires IMDB dataset with TEST_LABELS=True)")

Total misclassified: 2473 / 25000 (9.89%)

  #   Predicted      Actual  Text (first ~35 words)
-----------------------------------------------------------------------------------------------
  1    Positive    Negative  I saw this movie yesterday, and like most allrdy wrote "i also expected a Steven movie", god i love this guy just because his fighting style is unique and very humerous. In had a little...
  2    Negative    Positive  I thought this movie was really good. It ends up showing the viewers in the end that Leila should of kept what she had. Leila was sick of her husband Jim, who was more worried...
  3    Negative    Positive  "Comanche Moon" had everything going for it. For starters, Simon Wincer's back, a man who's name is synonymous with high-quality TV westerns. Unfortunately, the problems with "Moon" are something even the most talented director couldn't...
  4    Negative    Positive  That's a bad, raunchy, predictable, tacky, salacious soap opera? "The Best of Everyth

In [16]:
if DATASET == "IMDB" and TEST_LABELS and len(misclassified_idx) > 0:
    error_data = {
        "model": MODEL,
        "dataset": DATASET,
        "run_name": run_name,
        "misclassified_indices": misclassified_idx.tolist(),
        "true_labels": y_test[misclassified_idx].tolist(),
        "predicted_labels": y_test_pred[misclassified_idx].tolist(),
    }
    error_path = os.path.join(output_dir, "misclassified_indices.json")
    with open(error_path, "w", encoding="utf-8") as f:
        json.dump(error_data, f, indent=2, ensure_ascii=False)
    print(f"Misclassified indices saved: {error_path}")
    print(f"Total errors: {len(misclassified_idx)}")
else:
    print("No misclassified indices to save.")

Misclassified indices saved: c:\Users\cemil\OneDrive\Desktop\NLP\SVM\IMDB\C1_iters200_loss-squared_hinge_14_04_2026_14_42_29\misclassified_indices.json
Total errors: 2473


In [17]:
if DATASET == "SST-2":
    print("Pregatim arhiva oficiala pentru GLUE Benchmark...")

    # Folder temporar pentru generarea TSV-urilor
    tmp_dir = tempfile.mkdtemp()

    # Predictiile reale pentru SST-2
    sst2_path = os.path.join(tmp_dir, "SST-2.tsv")
    pd.DataFrame({
        "index": range(len(y_test_pred)),
        "prediction": y_test_pred
    }).to_csv(sst2_path, sep="\t", index=False)
    print(f"  SST-2.tsv: {len(y_test_pred)} predictii reale salvate.")

    # Fisiere dummy cu numarul corect de sample-uri per task
    # Label-uri corecte conform formatul oficial GLUE
    dummy_tasks = {
        "CoLA":    (1063,  "0"),                # 0 / 1
        "MRPC":    (1725,  "0"),                # 0 / 1
        "QQP":     (390965, "0"),               # 0 / 1
        "STS-B":   (1379,  "0.0"),              # regresie 0.0-5.0
        "MNLI-m":  (9796,  "entailment"),       # entailment / neutral / contradiction
        "MNLI-mm": (9847,  "entailment"),       # entailment / neutral / contradiction
        "QNLI":    (5463,  "entailment"),       # entailment / not_entailment
        "RTE":     (3000,  "entailment"),       # entailment / not_entailment
        "WNLI":    (146,   "0"),                # 0 / 1
        "AX":      (1104,  "entailment"),       # entailment / neutral / contradiction
    }

    for task, (n_samples, default_label) in dummy_tasks.items():
        dummy_path = os.path.join(tmp_dir, f"{task}.tsv")
        pd.DataFrame({
            "index": range(n_samples),
            "prediction": [default_label] * n_samples
        }).to_csv(dummy_path, sep="\t", index=False)
        print(f"  {task}.tsv: {n_samples} dummy predictii (label={default_label})")

    # Arhivam totul direct in output_dir
    zip_path = os.path.join(output_dir, "submission.zip")
    with zipfile.ZipFile(zip_path, "w") as zipf:
        for file in os.listdir(tmp_dir):
            zipf.write(os.path.join(tmp_dir, file), arcname=file)

    # Stergem folderul temporar
    shutil.rmtree(tmp_dir)

    print(f"\n{'='*60}")
    print(f"  Arhiva GLUE salvata in:")
    print(f"  {zip_path}")
    zip_size = os.path.getsize(zip_path)
    print(f"  Dimensiune: {zip_size:,} bytes")
    print(f"{'='*60}")
else:
    print(f"GLUE submission: skipped (DATASET={DATASET}, nu SST-2)")

GLUE submission: skipped (DATASET=IMDB, nu SST-2)
